In [104]:
import re
import torch
import torch.nn as nn
from typing import List
from sklearn.metrics import f1_score, classification_report

In [53]:
# Take BxT inputs. Say B = 69, T = 5
# Basically 69 sentences each of length 5

# Then we multiply it with the embedding dimension. Basically pass it through
# nn.Embedding(vocab_size, embedding_dim)

# Say we call the emdedding dim as C, say 768
# Now we have a tensor of size - B X T x C , after converting all the tokens to embeddings. Each sentence now has 5 words, each of length 768.

# We take the average of the all the 5 words, to get the average meaning of all the words. This is called a Bag of Words.
# Once we take the average we are left with a 1x768 tensor for each sentence.
# A single embedding for the whole sentence.

# So the tensor size for all sentences is Bx1xC, ie 69x1x768
# We write it as BxC, ie 69x768.

# Once we have a representation of a sentence in terms of an embedding vector of size 768, we can now pass it through a linear layer to reduce the 768 dimensions to only 1 number that will say whether the sentiment was positive or negative.

# We could have added more layers in between the 768 to 1 conversion, but for sake of simplicuty, we just use 1.

# So now our size becomes Bx1. ie 69x1, ie, 1 number for each sentence.

In [54]:
# Basic Model based on the above desciption
class SentimentAnalysis(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim) # say vocab size = 444, then this is 444 x768
        self.linear = nn.Linear(embedding_dim, 1)
        self.sigmoid = nn.Sigmoid()
        # In pytorch model contructor we don't include information about the batch size but we include batch size information in forward method.
    def forward(self, X):
        # X shape: [B, T] (Batch size x Sequence length)
        embedded = self.embedding(X) # This gives a vector of dim B x T x C
        avg = torch.mean(embedded, dim = 1) # This gives a vector of size B x C
        res = self.sigmoid(self.linear(avg))
        return res

In [55]:
# More sophisticated model.
# Improvement 1 - while doing torch.mean the 0 pads affect the Average value, bringing it down. I have to make sure to get the last real token in each setence while taking the average. Change the forward method
class SentimentAnalysisv2(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim) # say vocab size = 444, then this is 444 x768
        self.linear = nn.Linear(embedding_dim, 1)
        self.sigmoid = nn.Sigmoid()
        # In pytorch model contructor we don't include information about the batch size but we include batch size information in forward method.
    def forward(self, X):
        # X shape: [B, T]
        embedded = self.embedding(X)  # [B, T, C]

        # Create mask: 1 for real tokens, 0 for padding
        mask = (X != 0).float()  # [B, T]
        mask = mask.unsqueeze(2)  # [B, T, 1] — so it broadcasts with [B, T, C]

        # Zero out pad embeddings and average only real tokens
        masked_embedded = embedded * mask  # pads become zero vectors
        avg = masked_embedded.sum(dim=1) / mask.sum(dim=1)  # [B, C]

        res = self.sigmoid(self.linear(avg))
        return res

In [56]:
# To test this model I created a batch of 69 sentences.
file_path = '/content/sample_data/sentiment_dataset_1000.txt'

In [57]:
class DataPreprocessing:
    def __init__(self, file_path: str, max_len: int = 10):
        # Calling the 2 methods defined inside this class.

        sentences, labels = self.get_dataset(file_path)
        self.token_map, self.X = self.tokenize_dataset(sentences, max_len)

        # This self.token_map is a object instance level attribute. This is diffrent from the local variable token_map defined inside the tokenize_dataset. That method returns that local variable and here we capture that local variable as an object attribute.

        self.y = torch.tensor(labels, dtype=torch.float32)
        # Don't write self.token_map = {}, here. This is already being defined in the call to self.tokenize_dataset(). Having this here will replce the value created by tokenize_dataset with an empty dictionary.

    def get_dataset(self, file_path):
        # parse file, return (list_of_sentences, list_of_labels)
        with open(file_path, 'r', encoding='utf-8') as f:
            next(f)  # Skip header
            data = [line.strip().split('\t') for line in f if line.strip()]
            # line.strip() gives one string per row of f. Then, .split() is applied on each string. This gives a list for each string.
            # So from list of strings, we go to list of lists. Where the inner list has 2 elements, a sentence and a label.
        sentences, labels = zip(*data) # sentences and labels are tuples of length 69 now.
        sentences = [re.sub(r'[^\w\s]', '', s).lower() for s in sentences] #Remove all punctuations from sentences and convert everything to lowercase.
        labels = [int(label) for label in labels] # labels is converted into a list
        return sentences, labels

    def tokenize_dataset(self, sentences, max_len):
        # your existing logic, but also return the vocab
        # return token_map, padded_tensor

        # All sentences is list of tuple. Each tuple has a sentence and a label.
        # 1. Build vocabulary: collect all unique words, sort them, assign integer IDs starting at 1
        vocab = set()
        splitted_sentences = [sentence.split() for sentence in sentences] # Create a list of lists from the sentences list. Each element of inner list is now a word.
        for splitted_sentence in splitted_sentences:
            for word in splitted_sentence:
                vocab.add(word)
        sorted_vocab = sorted(list(vocab))
        token_map = {word: i for i, word in enumerate(sorted_vocab, start = 2)}
        token_map['<PAD>'] = 0 # ) for padding
        token_map["<UNK>"] = 1 # For unknown words.
        # Note: this token_map is a local variable different from the class attribute self.token_map. That will contian the value of this local variable once we return this local variable in the return statement.

        # 2. Encode each sentence by replacing words with their IDs
        output = []
        for splitted_sentence in splitted_sentences:
            tokenized_sentence = []
            for word in splitted_sentence:
                tokenized_sentence.append(token_map[word])
            # Truncate if too long
            tokenized_sentence = tokenized_sentence[:max_len] # max_len is T
            # Pad if too short
            tokenized_sentence += [0] * (max_len - len(tokenized_sentence))
            output.append(torch.tensor(tokenized_sentence))
        # 4. Pad shorter sequences with 0s using nn.utils.rnn.pad_sequence(tensors, batch_first=True)
        # 'output' now a python list of 1D tensors. To make it a proper 2D tensor, we use torch.stack(output). This gives us [B, T], ie, 69x10
        return token_map, torch.stack(output)

In [58]:
dataset = DataPreprocessing(file_path)
print(dataset.X.shape)
print(dataset.y.shape)
print(dataset.token_map)

torch.Size([1000, 10])
torch.Size([1000])
{'a': 2, 'about': 3, 'above': 4, 'absolutely': 5, 'absorbent': 6, 'access': 7, 'accessibility': 8, 'accessible': 9, 'accommodation': 10, 'accurate': 11, 'accurately': 12, 'acting': 13, 'action': 14, 'activities': 15, 'activity': 16, 'add': 17, 'addictive': 18, 'adds': 19, 'adjust': 20, 'adjustable': 21, 'admire': 22, 'adventure': 23, 'adventurous': 24, 'advertised': 25, 'affogato': 26, 'after': 27, 'afternoon': 28, 'again': 29, 'against': 30, 'ages': 31, 'air': 32, 'airport': 33, 'airy': 34, 'al': 35, 'alarm': 36, 'algae': 37, 'alive': 38, 'all': 39, 'almond': 40, 'alpine': 41, 'am': 42, 'amateurish': 43, 'amazed': 44, 'amazing': 45, 'ambiance': 46, 'ambient': 47, 'amenities': 48, 'an': 49, 'ancient': 50, 'and': 51, 'angle': 52, 'animation': 53, 'annoying': 54, 'any': 55, 'anyone': 56, 'anything': 57, 'anywhere': 58, 'apart': 59, 'app': 60, 'apple': 61, 'application': 62, 'appreciate': 63, 'appreciated': 64, 'aquarium': 65, 'are': 66, 'area': 6

In [59]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(dataset.X, dataset.y, test_size = 0.2, stratify=dataset.y, random_state = 42)

In [60]:
# I haven't taken care of the length of the input sequence at test or inference time. I have not decided, whether to cut them off during inference time if they are longer than the max length used during training

# Also while averaging the 0 pads affect the Average value, bringing it down. I have to make sure to get the last real token in each setence while taking the average.

In [61]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

torch.Size([800, 10])
torch.Size([800])
torch.Size([200, 10])
torch.Size([200])


In [99]:
# --- Inference on unseen data (negative-heavy dataset) ---
# This dataset was NOT used for training. The model has never seen these sentences.
# We tokenize using the TRAINING token_map so that words map to the same IDs the model learned.
# Words not in the training vocabulary get mapped to <UNK> (ID = 1).

def tokenize_for_inference(sentences, token_map, max_len=10):
    """Tokenize new sentences using the training vocabulary's token_map."""
    output = []
    for sentence in sentences:
        token_ids = [token_map.get(word, token_map["<UNK>"]) for word in sentence.split()]
        token_ids = token_ids[:max_len]
        token_ids += [0] * (max_len - len(token_ids))
        output.append(torch.tensor(token_ids))
    return torch.stack(output)

In [100]:
# Load the inference dataset (75% negative, 25% positive)
inference_file = 'sample_data/sentiment_dataset_200_neg_heavy.txt'
with open(inference_file, 'r', encoding='utf-8') as f:
    next(f)  # Skip header
    data = [line.strip().split('\t') for line in f if line.strip()]
inf_sentences, inf_labels = zip(*data)
inf_sentences = [re.sub(r'[^\w\s]', '', s).lower() for s in inf_sentences]
inf_labels = torch.tensor([int(label) for label in inf_labels], dtype=torch.float32)

In [97]:
# Tokenize using the TRAINING token_map — not building a new vocabulary
inf_X = tokenize_for_inference(inf_sentences, dataset.token_map)


In [105]:
# Run the batch through SentimentAnalysis model
embedding_dim = 768
model = SentimentAnalysisv2(len(dataset.token_map), embedding_dim)

loss_function = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

In [106]:
# Training Loop with metric tracking
epochs = 100
log_every = 10  # Log metrics every N epochs

# Lists to store metrics per logged epoch
history = {
    'epoch': [],
    'train_loss': [],
    'train_acc': [], 'train_f1': [],
    'test_acc': [], 'test_f1': [],
    'inf_acc': [], 'inf_f1': [],
}

for epoch in range(epochs):
    # --- Training step ---
    optimizer.zero_grad()
    model_prediction = model(X_train)
    loss = loss_function(model_prediction.squeeze(), y_train)
    loss.backward()
    optimizer.step()

    # --- Log metrics every N epochs ---
    if (epoch + 1) % log_every == 0:
        model.eval()
        with torch.no_grad():
            # Train metrics
            train_preds = (model_prediction.squeeze() >= 0.5).float()
            train_acc = (train_preds == y_train).float().mean().item()
            train_f1 = f1_score(y_train, train_preds, average='weighted', zero_division=0)

            # Test metrics
            test_preds = (model(X_test).squeeze() >= 0.5).float()
            test_acc = (test_preds == y_test).float().mean().item()
            test_f1 = f1_score(y_test, test_preds, average='weighted', zero_division=0)

            # Inference metrics (negative-heavy unseen data)
            inf_preds = (model(inf_X).squeeze() >= 0.5).float()
            inf_acc = (inf_preds == inf_labels).float().mean().item()
            inf_f1 = f1_score(inf_labels, inf_preds, average='weighted', zero_division=0)

        history['epoch'].append(epoch + 1)
        history['train_loss'].append(loss.item())
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        history['test_acc'].append(test_acc)
        history['test_f1'].append(test_f1)
        history['inf_acc'].append(inf_acc)
        history['inf_f1'].append(inf_f1)

        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | "
              f"Train Acc: {train_acc:.2%} F1: {train_f1:.2%} | "
              f"Test Acc: {test_acc:.2%} F1: {test_f1:.2%} | "
              f"Inf Acc: {inf_acc:.2%} F1: {inf_f1:.2%}")
        model.train()

Epoch  10 | Loss: 0.2484 | Train Acc: 93.88% F1: 93.75% | Test Acc: 81.50% F1: 80.31% | Inf Acc: 77.00% F1: 78.54%
Epoch  20 | Loss: 0.0284 | Train Acc: 99.87% F1: 99.88% | Test Acc: 83.00% F1: 81.62% | Inf Acc: 81.00% F1: 82.22%
Epoch  30 | Loss: 0.0045 | Train Acc: 100.00% F1: 100.00% | Test Acc: 85.00% F1: 84.41% | Inf Acc: 81.00% F1: 82.24%
Epoch  40 | Loss: 0.0013 | Train Acc: 100.00% F1: 100.00% | Test Acc: 84.00% F1: 83.37% | Inf Acc: 80.00% F1: 81.32%
Epoch  50 | Loss: 0.0007 | Train Acc: 100.00% F1: 100.00% | Test Acc: 84.00% F1: 83.37% | Inf Acc: 80.00% F1: 81.32%
Epoch  60 | Loss: 0.0005 | Train Acc: 100.00% F1: 100.00% | Test Acc: 84.00% F1: 83.37% | Inf Acc: 79.50% F1: 80.86%
Epoch  70 | Loss: 0.0004 | Train Acc: 100.00% F1: 100.00% | Test Acc: 84.00% F1: 83.37% | Inf Acc: 79.50% F1: 80.86%
Epoch  80 | Loss: 0.0004 | Train Acc: 100.00% F1: 100.00% | Test Acc: 84.00% F1: 83.37% | Inf Acc: 79.50% F1: 80.86%
Epoch  90 | Loss: 0.0003 | Train Acc: 100.00% F1: 100.00% | Test Acc

In [102]:
# --- Final evaluation with full classification reports ---
print("\n--- Final Test Set Evaluation ---")
with torch.no_grad():
    predictions = model(X_test).squeeze()
    predicted_labels = (predictions >= 0.5).float()
    accuracy = (predicted_labels == y_test).float().mean()
    print(f"Test accuracy: {accuracy.item():.2%}")
    print(f"F1 score: {f1_score(y_test, predicted_labels, average='weighted'):.2%}")
    print(classification_report(y_test, predicted_labels, target_names=['Negative', 'Positive']))


--- Final Test Set Evaluation ---
Test accuracy: 83.00%
F1 score: 82.48%
              precision    recall  f1-score   support

    Negative       0.69      0.58      0.63        50
    Positive       0.87      0.91      0.89       150

    accuracy                           0.83       200
   macro avg       0.78      0.75      0.76       200
weighted avg       0.82      0.83      0.82       200



In [103]:
# --- Final Inference Evaluation on negative-heavy dataset ---
print("\n--- Final Inference on negative-heavy dataset (200 sentences) ---")
with torch.no_grad():
    inf_predictions = model(inf_X).squeeze()
    inf_predicted_labels = (inf_predictions >= 0.5).float()
    inf_accuracy = (inf_predicted_labels == inf_labels).float().mean()
    print(f"Accuracy: {inf_accuracy.item():.2%}")
    print(f"F1 score: {f1_score(inf_labels, inf_predicted_labels, average='weighted'):.2%}")
    print(classification_report(inf_labels, inf_predicted_labels, target_names=['Negative', 'Positive']))



--- Final Inference on negative-heavy dataset (200 sentences) ---
Accuracy: 70.50%
F1 score: 72.36%
              precision    recall  f1-score   support

    Negative       1.00      0.61      0.76       150
    Positive       0.46      1.00      0.63        50

    accuracy                           0.70       200
   macro avg       0.73      0.80      0.69       200
weighted avg       0.86      0.70      0.72       200

